# ASL Citizen - 300 Class Stable Training (RGB Frame Model)

نسخة مبنية على أفضل baseline حقق أفضل نتيجة عند 20 class، لكن متعدلة للتدريب على 300 class بشكل أكثر ثباتًا على Kaggle.

أهم التعديلات:
- اختيار **300 classes مشتركة** بين train/val/test بدل top-K من train فقط
- تشغيل على **الداتا الكاملة** بدون sample mode
- **WeightedRandomSampler** فقط لعلاج عدم الاتزان
- **10 epochs total** = 4 في Stage 1 + 6 في Stage 2
- **optional warm start** من checkpoint قديم لو موجود


In [ ]:
import os
import random
import warnings
from collections import Counter

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [ ]:
import os
from pathlib import Path
import pandas as pd

BASE_INPUT = Path("/kaggle/input")

# دور على ملفات train/val/test.csv تلقائيًا
train_candidates = list(BASE_INPUT.rglob("train.csv"))
val_candidates   = list(BASE_INPUT.rglob("val.csv"))
test_candidates  = list(BASE_INPUT.rglob("test.csv"))

print("train candidates:", train_candidates[:5])
print("val candidates  :", val_candidates[:5])
print("test candidates :", test_candidates[:5])

TRAIN_CSV = str(train_candidates[0])
VAL_CSV   = str(val_candidates[0])
TEST_CSV  = str(test_candidates[0])

# غالبًا فولدر splits
SPLITS_DIR = str(Path(TRAIN_CSV).parent)

# دور على فولدر الفيديوهات
video_dir_candidates = [p for p in BASE_INPUT.rglob("*") if p.is_dir() and "video" in p.name.lower()]
print("video dir candidates:", video_dir_candidates[:10])

VIDEOS_DIR = str(video_dir_candidates[0])

VIDEO_COL = "Video file"
LABEL_COL = "Gloss"

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

def add_paths(df):
    df = df.copy()
    df["video_path"] = df[VIDEO_COL].apply(lambda x: os.path.join(VIDEOS_DIR, x))
    df = df[df["video_path"].apply(os.path.exists)].reset_index(drop=True)
    return df

train_df = add_paths(train_df)
val_df   = add_paths(val_df)
test_df  = add_paths(test_df)

print("TRAIN_CSV:", TRAIN_CSV)
print("VAL_CSV  :", VAL_CSV)
print("TEST_CSV :", TEST_CSV)
print("VIDEOS_DIR:", VIDEOS_DIR)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))
print("Unique classes in train:", train_df[LABEL_COL].nunique())

train candidates: [PosixPath('/kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/splits/train.csv')]
val candidates  : [PosixPath('/kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/splits/val.csv')]
test candidates : [PosixPath('/kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/splits/test.csv')]
video dir candidates: [PosixPath('/kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/videos')]
TRAIN_CSV: /kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/splits/train.csv
VAL_CSV  : /kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/splits/val.csv
TEST_CSV : /kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/splits/test.csv
VIDEOS_DIR: /kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/videos
Train: 40154
Val  : 10304
Test : 32941
Unique classes in train: 2731


In [ ]:
TOP_K_CLASSES = 200
USE_SAMPLE_MODE = False
TRAIN_SAMPLE_PER_CLASS = 180
VAL_SAMPLE_PER_CLASS   = 50
TEST_SAMPLE_PER_CLASS  = 50

IMG_SIZE = 224
NUM_FRAMES = 8
BATCH_SIZE = 8
NUM_WORKERS = 0

STAGE1_EPOCHS = 40
STAGE2_EPOCHS = 0
PATIENCE = 3

LR_STAGE1 = 7e-4
LR_STAGE2 = 7e-5
WEIGHT_DECAY = 3e-4
LABEL_SMOOTHING = 0.1
DROPOUT = 0.60

BEST_PATH = "/kaggle/working/asl_rgb_200_best.pth"
LAST_PATH = "/kaggle/working/asl_rgb_200_last.pth"

BEST_PATH_20 = "/kaggle/input/models/motarek122112/grade-1-11/other/default/1/asl_rgb_best.pth"
LAST_PATH_20 = "/kaggle/input/models/motarek122112/grade-1-11/other/default/1/asl_rgb_last.pth"

WARMSTART_CKPT_CANDIDATES = [
    BEST_PATH_20,
    LAST_PATH_20,
]

In [ ]:
# =========================
# Keep top-K COMMON classes across train/val/test
# =========================
def normalize_label(x):
    return str(x).strip().upper()

train_df[LABEL_COL] = train_df[LABEL_COL].apply(normalize_label)
val_df[LABEL_COL]   = val_df[LABEL_COL].apply(normalize_label)
test_df[LABEL_COL]  = test_df[LABEL_COL].apply(normalize_label)

train_counts = train_df[LABEL_COL].value_counts()
val_classes = set(val_df[LABEL_COL].unique().tolist())
test_classes = set(test_df[LABEL_COL].unique().tolist())

common_ranked_classes = [
    cls for cls in train_counts.index
    if cls in val_classes and cls in test_classes
]

selected_classes = common_ranked_classes[:TOP_K_CLASSES]

train_top = train_df[train_df[LABEL_COL].isin(selected_classes)].copy()
val_top   = val_df[val_df[LABEL_COL].isin(selected_classes)].copy()
test_top  = test_df[test_df[LABEL_COL].isin(selected_classes)].copy()

print("Selected common classes:", len(selected_classes))
print("Train top:", len(train_top))
print("Val top  :", len(val_top))
print("Test top :", len(test_top))

print("\nTop 10 selected classes:")
for cls in selected_classes[:10]:
    print(
        cls,
        "| train:", int((train_top[LABEL_COL] == cls).sum()),
        "| val:", int((val_top[LABEL_COL] == cls).sum()),
        "| test:", int((test_top[LABEL_COL] == cls).sum()),
    )


Selected common classes: 200
Train top: 3437
Val top  : 697
Test top : 2449

Top 10 selected classes:
DOG1 | train: 24 | val: 4 | test: 17
HURDLE/TRIP1 | train: 22 | val: 3 | test: 13
DEMAND1 | train: 21 | val: 4 | test: 14
BITE1 | train: 21 | val: 4 | test: 14
DARK1 | train: 21 | val: 3 | test: 15
BREAKFAST1 | train: 21 | val: 3 | test: 15
MECHANIC1 | train: 20 | val: 3 | test: 16
PARTY1 | train: 20 | val: 4 | test: 15
ROCKINGCHAIR1 | train: 20 | val: 4 | test: 15
DEAF1 | train: 20 | val: 4 | test: 15


In [ ]:
def stratified_cap(df, label_col, per_class, seed=42):
    parts = []
    for label, grp in df.groupby(label_col):
        n = min(per_class, len(grp))
        parts.append(grp.sample(n=n, random_state=seed))
    out = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

if USE_SAMPLE_MODE:
    train_sub = stratified_cap(train_top, LABEL_COL, TRAIN_SAMPLE_PER_CLASS, seed=SEED)
    val_sub   = stratified_cap(val_top, LABEL_COL, VAL_SAMPLE_PER_CLASS, seed=SEED)
    test_sub  = stratified_cap(test_top, LABEL_COL, TEST_SAMPLE_PER_CLASS, seed=SEED)
else:
    train_sub = train_top.sample(frac=1, random_state=SEED).reset_index(drop=True)
    val_sub   = val_top.sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_sub  = test_top.sample(frac=1, random_state=SEED).reset_index(drop=True)

common_labels = (
    set(train_sub[LABEL_COL].unique()) &
    set(val_sub[LABEL_COL].unique()) &
    set(test_sub[LABEL_COL].unique())
)

train_sub = train_sub[train_sub[LABEL_COL].isin(common_labels)].reset_index(drop=True)
val_sub   = val_sub[val_sub[LABEL_COL].isin(common_labels)].reset_index(drop=True)
test_sub  = test_sub[test_sub[LABEL_COL].isin(common_labels)].reset_index(drop=True)

label_names = sorted(train_sub[LABEL_COL].unique().tolist())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}

train_sub["label_id"] = train_sub[LABEL_COL].map(label2id)
val_sub["label_id"]   = val_sub[LABEL_COL].map(label2id)
test_sub["label_id"]  = test_sub[LABEL_COL].map(label2id)

num_classes = len(label_names)

print("num_classes:", num_classes)
print("train_sub:", len(train_sub))
print("val_sub  :", len(val_sub))
print("test_sub :", len(test_sub))
print("train label min/max counts:", int(train_sub[LABEL_COL].value_counts().min()), int(train_sub[LABEL_COL].value_counts().max()))


num_classes: 200
train_sub: 3437
val_sub  : 697
test_sub : 2449
train label min/max counts: 16 24


In [ ]:
# Dataset
class ASLRGBDataset(Dataset):
    def __init__(self, df, num_frames=8, img_size=224, train_mode=False):
        self.df = df.reset_index(drop=True)
        self.num_frames = num_frames
        self.img_size = img_size
        self.train_mode = train_mode

    def __len__(self):
        return len(self.df)

    def _uniform_indices(self, total_frames):
        if total_frames <= 0:
            return [0] * self.num_frames
        if total_frames < self.num_frames:
            idxs = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        else:
            if self.train_mode:
                # jitter بسيط بدل crop عشوائي عنيف
                base = np.linspace(0, total_frames - 1, self.num_frames)
                noise = np.random.uniform(-0.5, 0.5, size=self.num_frames) * max(1, total_frames / (2 * self.num_frames))
                idxs = np.clip(np.round(base + noise), 0, total_frames - 1).astype(int)
                idxs = np.sort(idxs)
            else:
                idxs = np.linspace(0, total_frames - 1, self.num_frames).astype(int)
        return idxs.tolist()

    def _augment_frame(self, frame):
        # frame: RGB uint8
        if self.train_mode and np.random.rand() < 0.5:
            frame = cv2.flip(frame, 1)

        if self.train_mode:
            h, w = frame.shape[:2]

            # random crop خفيف
            scale = np.random.uniform(0.85, 1.0)
            nh, nw = int(h * scale), int(w * scale)
            y1 = np.random.randint(0, h - nh + 1) if h > nh else 0
            x1 = np.random.randint(0, w - nw + 1) if w > nw else 0
            frame = frame[y1:y1+nh, x1:x1+nw]

            # brightness / contrast
            alpha = np.random.uniform(0.9, 1.1)
            beta = np.random.uniform(-10, 10)
            frame = np.clip(alpha * frame + beta, 0, 255).astype(np.uint8)

        frame = cv2.resize(frame, (self.img_size, self.img_size))
        frame = frame.astype(np.float32) / 255.0

        # ImageNet normalization
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        frame = (frame - mean) / std

        frame = np.transpose(frame, (2, 0, 1))  # C,H,W
        return frame

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["video_path"]

        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()

        if len(frames) == 0:
            frames = [np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)]

        idxs = self._uniform_indices(len(frames))
        sampled = [frames[i] for i in idxs]
        sampled = [self._augment_frame(f) for f in sampled]

        x = np.stack(sampled, axis=0)  # T,C,H,W
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(int(row["label_id"]), dtype=torch.long)
        return x, y

In [ ]:
train_dataset = ASLRGBDataset(train_sub, num_frames=NUM_FRAMES, img_size=IMG_SIZE, train_mode=True)
val_dataset   = ASLRGBDataset(val_sub,   num_frames=NUM_FRAMES, img_size=IMG_SIZE, train_mode=False)
test_dataset  = ASLRGBDataset(test_sub,  num_frames=NUM_FRAMES, img_size=IMG_SIZE, train_mode=False)

class_counts = train_sub["label_id"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = train_sub["label_id"].map(lambda x: class_weights[x]).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

Train batches: 430
Val batches  : 88
Test batches : 307


In [ ]:
# Model
class TemporalResNet(nn.Module):
    def __init__(self, num_classes, dropout=0.35):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, t, -1)
        feats = feats.mean(dim=1)         # temporal average
        feats = self.dropout(feats)
        out = self.head(feats)
        return out

model = TemporalResNet(num_classes=num_classes, dropout=DROPOUT).to(device)

loaded_warmstart = None
for ckpt_path in WARMSTART_CKPT_CANDIDATES:
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        if isinstance(ckpt, dict) and "model_state" in ckpt:
            ckpt = ckpt["model_state"]
        model_dict = model.state_dict()
        compatible = {k: v for k, v in ckpt.items() if k in model_dict and tuple(v.shape) == tuple(model_dict[k].shape)}
        if compatible:
            model_dict.update(compatible)
            model.load_state_dict(model_dict, strict=False)
            loaded_warmstart = ckpt_path
            break

print("Warm start checkpoint:", loaded_warmstart if loaded_warmstart else "None")

# Freeze everything first except layer4 + head
for p in model.backbone.parameters():
    p.requires_grad = False

for p in model.backbone.layer4.parameters():
    p.requires_grad = True

for p in model.head.parameters():
    p.requires_grad = True

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def trainable_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("Trainable params:", f"{trainable_params(model):,}")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 204MB/s]


Warm start checkpoint: None
Trainable params: 8,496,328


In [ ]:
def make_optimizer(lr):
    return optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=WEIGHT_DECAY
    )

optimizer = make_optimizer(LR_STAGE1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)

In [ ]:
def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_true = []
    all_pred = []

    pbar = tqdm(loader, leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = criterion(logits, y)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * y.size(0)

        all_true.extend(y.detach().cpu().numpy().tolist())
        all_pred.extend(preds.detach().cpu().numpy().tolist())

        acc_now = accuracy_score(all_true, all_pred)
        pbar.set_postfix({
            "loss": f"{total_loss / len(all_true):.4f}",
            "acc": f"{acc_now:.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
        })

    epoch_loss = total_loss / len(all_true)
    epoch_acc = accuracy_score(all_true, all_pred)
    epoch_f1 = f1_score(all_true, all_pred, average="macro")

    return epoch_loss, epoch_acc, epoch_f1, all_true, all_pred

In [ ]:
best_val_f1 = -1.0
epochs_no_improve = 0
history = []

def save_checkpoint(path, epoch, stage):
    torch.save({
        "epoch": epoch,
        "stage": stage,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "history": history,
        "label2id": label2id,
        "id2label": id2label
    }, path)

print("===== Stage 1: layer4 + head =====")
for epoch in range(1, STAGE1_EPOCHS + 1):
    train_loss, train_acc, train_f1, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_f1, _, _ = run_epoch(val_loader, train=False)

    scheduler.step(val_f1)

    history.append({
        "stage": 1,
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1
    })

    print(f"Epoch {epoch}/{STAGE1_EPOCHS}")
    print(f"Train -> loss: {train_loss:.4f} | acc: {train_acc:.4f} | f1: {train_f1:.4f}")
    print(f"Val   -> loss: {val_loss:.4f} | acc: {val_acc:.4f} | f1: {val_f1:.4f}")

    save_checkpoint(LAST_PATH, epoch, 1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_PATH)
        print("Saved BEST model.")
    else:
        epochs_no_improve += 1
        print(f"No improvement: {epochs_no_improve}")

    if epochs_no_improve >= PATIENCE:
        print("Early stopping in Stage 1")
        break

===== Stage 1: layer4 + head =====


  0%|          | 0/430 [00:00<?, ?it/s]

  0%|          | 0/88 [00:00<?, ?it/s]

Epoch 1/40
Train -> loss: 5.6134 | acc: 0.0044 | f1: 0.0038
Val   -> loss: 5.3337 | acc: 0.0072 | f1: 0.0008
Saved BEST model.


  0%|          | 0/430 [00:00<?, ?it/s]

...

  0%|          | 0/88 [00:00<?, ?it/s]

Epoch 36/40
Train -> loss: 1.2612 | acc: 0.9799 | f1: 0.9795
Val   -> loss: 2.7277 | acc: 0.4806 | f1: 0.4719
No improvement: 2


  0%|          | 0/430 [00:00<?, ?it/s]

  0%|          | 0/88 [00:00<?, ?it/s]

Epoch 37/40
Train -> loss: 1.2321 | acc: 0.9860 | f1: 0.9855
Val   -> loss: 2.6957 | acc: 0.4864 | f1: 0.4760
No improvement: 3
Early stopping in Stage 1


In [ ]:
# Stage 2: unfreeze layer3
for p in model.backbone.layer3.parameters():
    p.requires_grad = True

optimizer = make_optimizer(LR_STAGE2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1,
)

epochs_no_improve = 0

print("===== Stage 2: layer3 + layer4 + head =====")
for epoch in range(1, STAGE2_EPOCHS + 1):
    train_loss, train_acc, train_f1, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_f1, _, _ = run_epoch(val_loader, train=False)

    scheduler.step(val_f1)

    history.append({
        "stage": 2,
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1
    })

    print(f"Epoch {epoch}/{STAGE2_EPOCHS}")
    print(f"Train -> loss: {train_loss:.4f} | acc: {train_acc:.4f} | f1: {train_f1:.4f}")
    print(f"Val   -> loss: {val_loss:.4f} | acc: {val_acc:.4f} | f1: {val_f1:.4f}")

    save_checkpoint(LAST_PATH, STAGE1_EPOCHS + epoch, 2)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_PATH)
        print("Saved BEST model.")
    else:
        epochs_no_improve += 1
        print(f"No improvement: {epochs_no_improve}")

    if epochs_no_improve >= PATIENCE:
        print("Early stopping in Stage 2")
        break

print("Best validation macro-F1:", round(best_val_f1, 4))

===== Stage 2: layer3 + layer4 + head =====
Best validation macro-F1: 0.4984


In [ ]:
hist_df = pd.DataFrame(history)
hist_df

,stage,epoch,train_loss,train_acc,train_f1,val_loss,val_acc,val_f1
0,1,1,5.613394,0.004364,0.003790,5.333732,0.007174,0.000772
1,1,2,5.348605,0.013675,0.008917,5.199986,0.011478,0.004756
2,1,3,5.032487,0.028222,0.019511,5.035083,0.027260,0.008112
3,1,4,4.689327,0.052371,0.038678,4.919890,0.025825,0.016587
4,1,5,4.399053,0.075938,0.061963,4.516854,0.063128,0.029915
5,1,6,4.030368,0.132383,0.109164,4.329374,0.101865,0.065523
6,1,7,3.695752,0.210358,0.183593,4.156260,0.167862,0.114467
7,1,8,3.408840,0.272040,0.249491,3.804063,0.176471,0.150864
8,1,9,3.106041,0.353215,0.327103,3.816352,0.203730,0.160950
9,1,10,2.897500,0.420134,0.396539,3.662863,0.228121,0.205807


In [ ]:
# Load best model
if os.path.exists(BEST_PATH):
    model.load_state_dict(torch.load(BEST_PATH, map_location=device))
    print("Loaded best model from:", BEST_PATH)
else:
    print("Best model file not found.")

Loaded best model from: /kaggle/working/asl_rgb_200_best.pth


In [ ]:
# Validation detailed report
val_loss, val_acc, val_f1, y_true_val, y_pred_val = run_epoch(val_loader, train=False)

print("Validation Accuracy :", round(val_acc, 4))
print("Validation Macro-F1 :", round(val_f1, 4))
print()
print(classification_report(
    y_true_val,
    y_pred_val,
    target_names=[id2label[i] for i in range(num_classes)],
    digits=4,
    zero_division=0
))

  0%|          | 0/88 [00:00<?, ?it/s]

Validation Accuracy : 0.4964
Validation Macro-F1 : 0.4984

                        precision    recall  f1-score   support

                 8HOUR     0.6667    0.6667    0.6667         3
               ADDRESS     0.3333    0.2500    0.2857         4
             ADVERTISE     0.5000    0.3333    0.4000         3
           ALLOFSUDDEN     1.0000    0.3333    0.5000         3
                ANYONE     0.0000    0.0000    0.0000         3
                 APPLE     0.6667    1.0000    0.8000         4
        ARTICULATESIGN     0.3333    0.2500    0.2857         4
              ASSEMBLY     0.5000    0.2500    0.3333         4
               AUTISM1     0.0000    0.0000    0.0000         5
                  AXE1     1.0000    0.5000    0.6667         4
                 BABY2     1.0000    1.0000    1.0000         3
               BACKOUT     0.5000    0.2500    0.3333         4
             BACKPACK1     0.2500    0.3333    0.2857         3
               BANDAGE     1.0000    0.2500 

In [ ]:
# Test evaluation
test_loss, test_acc, test_f1, y_true_test, y_pred_test = run_epoch(test_loader, train=False)

print("Test Accuracy :", round(test_acc, 4))
print("Test Macro-F1 :", round(test_f1, 4))

  0%|          | 0/307 [00:00<?, ?it/s]

Test Accuracy : 0.4443
Test Macro-F1 : 0.4427


In [ ]:

# Sample predictions
model.eval()
num_show = min(50, len(test_dataset))

for i in range(num_show):
    x, y = test_dataset[i]
    with torch.no_grad():
        logits = model(x.unsqueeze(0).to(device))
        pred = logits.argmax(dim=1).item()

    true_name = id2label[y.item()]
    pred_name = id2label[pred]
    result = "Correct" if pred == y.item() else "Wrong"

    print(f"Sample {i+1}")
    print("True label     =", true_name)
    print("Predicted label=", pred_name)
    print("Result         =", result)
    print("-" * 50)

Sample 1
True label     = FOREIGNER1
Predicted label= CEMETERY
Result         = Wrong
--------------------------------------------------
Sample 2
True label     = DOWNSIZE1
Predicted label= VLOG
Result         = Wrong
--------------------------------------------------
Sample 3
True label     = HOW1
Predicted label= HOW1
Result         = Correct
--------------------------------------------------
Sample 4
True label     = SLICE1
Predicted label= IMPOSSIBLE
Result         = Wrong
--------------------------------------------------
Sample 5
True label     = AXE1
Predicted label= NOON1
Result         = Wrong
--------------------------------------------------
Sample 6
True label     = PIPE2
Predicted label= PIPE2
Result         = Correct
--------------------------------------------------
Sample 7
True label     = FINE1
Predicted label= BEE1
Result         = Wrong
--------------------------------------------------
Sample 8
True label     = TIEUP2
Predicted label= TIEUP2
Result         = Correc